# [LAB12] 딥러닝 > 신경망의 이해 > 05. 단순선형회귀

속도에 따른 제동거리 예측 데이터셋

## 📘 #01. 준비작업

### 📝 [1] 패키지 참조

In [ ]:
from hossam import *
from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import RootMeanSquaredError, R2Score
from tqdm.keras import TqdmCallback

### 📝 [2] 데이터셋 준비

## 📘 #02. 탐색적 데이터 분석

### 📝 [1] 데이터 품질 검사

In [ ]:
origin = load_data('cars')
origin.head()

In [ ]:
desc = origin.describe().T
num_cols = origin.select_dtypes(include=np.number).columns
for column in num_cols:
    skewness = origin[column].skew()
    if abs(skewness) < 0.5:
        strength = "week"
        log_transform = "not needed"
    elif abs(skewness) < 1:
        strength = "normal"
        log_transform = "recommended"
    else:
        strength = "strong"
        log_transform = "needed"
    desc.loc[column, "skewness"] = skewness
    desc.loc[column, "skewness_strength"] = strength
    desc.loc[column, "log_transform"] = log_transform
desc

### 📝 [2] 제동거리에 대한 로그변환

## 📘 #03. 데이터 전처리

### 📝 [1] 훈련/검증 데이터 분리

In [ ]:
df = origin.copy()
df['dist'] = np.log1p(df['dist'])
df.head()

In [ ]:
yname = "dist"
x = df.drop(columns=[yname])
y = df[yname]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=52)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
rows, cols = x_train.shape
print(rows, cols)

### 📝 [2] 데이터 스케일링

In [ ]:
scaler = StandardScaler()
scaler.fit(x_train)
x_train_scaled = DataFrame(scaler.transform(x_train), columns=x_train.columns)
x_test_scaled = DataFrame(scaler.transform(x_test), columns=x_test.columns)
display(x_train_scaled.head())
display(x_test_scaled.head())

## 📘 #04. 신경망 모델 적합

### 📝 [1] 신경망 정의

은닉층의 뉴런 수를 2의 배수만큼 변경해 가면서 최적의 은닉층 수를 찾아야 한다. → 하이퍼파라미터 튜닝

| 구분 | 모델 | 활성화 함수 | 옵티마이저 | 손실함수 | 평가지표 | 과적합 판정 지표 | 대표예제 |
|------|------|-------------|------------|----------|----------|------------------|----------|
| 회귀 | 단순선형회귀 | linear | Adam | mse | mae, rmse, R2 | Val RMSE − Train RMSE | 제동거리 예측 |

In [ ]:
model = Sequential()
model.add(Input(shape=(cols,)))
relu_units = 32
model.add(Dense(relu_units, activation="relu"))
model.add(Dense(1, activation="linear"))
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae", RootMeanSquaredError(name="rmse"), R2Score(name="r2")],
)
model.summary()

### 📝 [2] 학습하기

In [ ]:
%%time
result = model.fit(
    x_train, y_train,
    epochs=500,
    validation_data=(x_test, y_test),
    verbose=0,
    callbacks=[
        TqdmCallback(verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, min_delta=0.001),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0, verbose=1)
    ]
)
result

## 📘 #05 성능평가

### 📝 [1] 성능평가 지표

### 📝 [2] 학습 과정 확인

### 📝 [3] Loss, RMSE 학습곡선

In [ ]:
train_eval = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_eval = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
final_results = DataFrame([train_eval, test_eval])
final_results.insert(0, "Dataset", ["Train", "Test"])
final_results["RMSE_gap"] = None
final_results.loc[1, "RMSE_gap"] = final_results.loc[1, "rmse"] - final_results.loc[0, "rmse"]
final_results

In [ ]:
history_df = DataFrame(data=result.history)
history_df["epoch"] = history_df.index + 1
history_df.head()

In [ ]:
figsize = (1600 / 100, 600 / 100)
fig, ax = plt.subplots(1, 2, figsize=figsize, dpi=100)
fig.suptitle(f'Relu Rnits={relu_units}', fontsize=20, color='#006600')
fig.subplots_adjust(wspace=0.2, hspace=0.2)
sb.lineplot(data=history_df, x="epoch", y="loss", ax=ax[0], label="Train Loss")
sb.lineplot(data=history_df, x="epoch", y="val_loss", ax=ax[0], label="Validation Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_title("Training vs Validation Loss")
ax[0].grid(True, alpha=0.3)
sb.lineplot(data=history_df, x="epoch", y="rmse", ax=ax[1], label="Train RMSE")
sb.lineplot(data=history_df, x="epoch", y="val_rmse", ax=ax[1], label="Validation RMSE")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("RMSE")
ax[1].set_title("Training vs Validation RMSE")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"relu_units_{relu_units}_learning_curve.png", dpi=200)
plt.show()
plt.close()

## 📘 #06. 예측 결과 활용

### 📝 [1] 예측치 구하기

### 📝 [2] 결과 데이터 셋 구성

### 📝 [3] 관측치와 예측치 비교 시각화

In [ ]:
pred = model.predict(x_test, verbose=0)
pred

In [ ]:
kdf = DataFrame({
    '검증데이터': x_test['speed'],
    '실제값': y_test,
    '예측값': pred.flatten()
})
kdf['오차'] = kdf['실제값']-kdf['예측값']
kdf.head()

In [ ]:
figsize = (1280 / 100, 720 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=100)
sb.regplot(data=kdf, x='검증데이터', y='실제값', label='실제값')
sb.regplot(data=kdf, x='검증데이터', y='예측값', label='예측값')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title(f"실제값 vs 예측값 (relu_units={relu_units})")
plt.tight_layout()
plt.savefig(f"relu_units_{relu_units}_actual_vs_predicted.png", dpi=200)
plt.show()
plt.close()